# Notebook 07e — BSISO Supervised Embedding-Dimension Sweep
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

nb07d found the 2-D supervised encoder plateaus at **~50%** phase (vs 64-D **67.7%**) once the collapse is fixed (τ=0.07). The BSISO NSV chapter found intrinsic dimension **d̂ = 4** (Session 32). This notebook sweeps the embedding dimension **{1, 2, 4, 8, 16, 32, 64}** with the **locked recipe** from nb07d and tests the hypothesis:

> phase accuracy climbs steeply 2→4 and **saturates near d = 4**, giving an end-to-end *supervised* confirmation of the NSV intrinsic dimension.

**Locked recipe (from nb07d):** variant = `vicreg`, batch = 256, τ = 0.07, weight_decay = 1e-3, cosine LR, **full epochs (no early stopping** — nb07d showed ES on val loss stops before the embedding finishes spreading).

**Per-run metrics:** RMM/BSISO-phase linear probe, ENSO displacement z, generalized effective rank `(Σλ)²/Σλ²` of the d-D embedding covariance, norm max.

**Inputs** (`data/processed/`): `X_MJJAS_lee.npy` (N≈6579, 3, 31, 51), `labels_aligned_mjjas_lee.csv`.  
**Runtime:** 7 dims × 1 seed (curve) + 3-seed robustness at the best 2 dims. **Use a T4 GPU** (nb07d ran on CPU and was slow).

---

## Cell 1 — Setup + Generalized Embedding Metrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
RESULTS_DIR   = f'{PROJECT_DIR}/results/sup_dim_sweep'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('WARNING: no GPU — switch Colab runtime to T4 (Runtime > Change runtime type).')


def emb_metrics(Z):
    """Geometry diagnostics for a d-D embedding matrix Z (n, d)."""
    Z = np.asarray(Z, dtype=np.float64)
    norms = np.linalg.norm(Z, axis=1)
    cov = np.atleast_2d(np.cov(Z.T))
    ev = np.sort(np.linalg.eigvalsh(cov))[::-1]
    ev = np.clip(ev, 0, None)
    eff_rank = float((ev.sum() ** 2) / (np.sum(ev ** 2) + 1e-12))
    eig_ratio = float(ev[1] / (ev[0] + 1e-12)) if len(ev) > 1 else 0.0
    return {'eff_rank': eff_rank, 'eig_ratio': eig_ratio,
            'norm_mean': float(norms.mean()), 'norm_max': float(norms.max())}

print('Metrics ready.')

## Cell 2 — Load Data + Year Split + Phase-ENSO Index

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/X_MJJAS_lee.npy').astype(np.float32)
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv', parse_dates=['date'])
print(f'X shape: {X.shape}   labels: {len(labels)} rows')
assert X.shape[0] == len(labels)

all_years   = sorted(labels['date'].dt.year.unique())
val_years   = all_years[::5]
train_years = [y for y in all_years if y not in val_years]
year_col    = labels['date'].dt.year
train_idx   = labels.index[year_col.isin(train_years)].values
val_idx     = labels.index[year_col.isin(val_years)].values
print(f'Train: {len(train_idx)}   Val: {len(val_idx)}')

phase_enso_index = defaultdict(list)
for idx in train_idx:
    row = labels.loc[idx]
    if row['bsiso_amplitude'] > 1.0:
        phase_enso_index[(row['bsiso_phase'], row['enso_category'])].append(idx)
print(f'Active train bins: {len(phase_enso_index)}  ({sum(len(v) for v in phase_enso_index.values())} days)')

active_mask_all = (labels['bsiso_amplitude'].values > 1.0)
val_active_idx  = np.intersect1d(val_idx, np.where(active_mask_all)[0])

## Cell 3 — Sampler + Dataset (same pairing as nb07c/07d)

In [ ]:
class PairSampler:
    def __init__(self, labels_df, index):
        self.labels = labels_df; self.index = index
        self.enso_categories = labels_df['enso_category'].unique().tolist()
    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]
    def sample_positive_pair(self):
        key = self._random_category(); ind = self.index[key]
        if len(ind) < 2: return self.sample_easy_negative_pair()
        a, b = np.random.choice(ind, size=2, replace=False)
        yA = self.labels.loc[a, 'date'].year
        other = [i for i in ind if self.labels.loc[i, 'date'].year != yA]
        if other: b = np.random.choice(other)
        return a, b
    def sample_hard_negative_pair(self):
        if len(self.enso_categories) < 2: return self.sample_easy_negative_pair()
        ph = np.random.choice(range(1, 9)); eA, eB = np.random.choice(self.enso_categories, 2, replace=False)
        kA, kB = (ph, eA), (ph, eB)
        if not self.index[kA] or not self.index[kB]: return self.sample_positive_pair()
        return np.random.choice(self.index[kA]), np.random.choice(self.index[kB])
    def sample_easy_negative_pair(self):
        pA, pB = np.random.choice(range(1, 9), 2, replace=False)
        eA = np.random.choice(self.enso_categories); eB = np.random.choice(self.enso_categories)
        kA, kB = (pA, eA), (pB, eB)
        a = (np.random.choice(self.index[kA]) if self.index[kA]
             else self.labels[self.labels['bsiso_phase'] == pA].sample(1).index[0])
        b = (np.random.choice(self.index[kB]) if self.index[kB]
             else self.labels[self.labels['bsiso_phase'] == pB].sample(1).index[0])
        return a, b


class PairDataset(Dataset):
    def __init__(self, X_data, labels_df, index, mode, indices):
        self.X = X_data; self.labels = labels_df
        self.sampler = PairSampler(labels_df, index); self.mode = mode; self.indices = indices
        if mode == 'val': self.val_pairs = self._val_pairs()
    def _val_pairs(self):
        pairs = []; vl = self.labels.loc[self.indices]
        for ph in range(1, 9):
            for en in self.sampler.enso_categories:
                g = vl[(vl['bsiso_phase'] == ph) & (vl['enso_category'] == en)].index.tolist()
                for i in range(len(g)):
                    for j in range(i + 1, len(g)): pairs.append((g[i], g[j]))
        return pairs[:1000]
    def __len__(self):
        return len(self.indices) if self.mode == 'train' else len(self.val_pairs)
    def __getitem__(self, i):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:   a, b = self.sampler.sample_positive_pair()
            elif r < 0.50: a, b = self.sampler.sample_hard_negative_pair()
            else:          a, b = self.sampler.sample_easy_negative_pair()
        else:
            a, b = self.val_pairs[i]
        return torch.from_numpy(self.X[a]).float(), torch.from_numpy(self.X[b]).float()

print('Dataset ready.')

## Cell 4 — Parameterized Encoder + VICReg Loss

In [ ]:
class SupEncoder(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 32, 3, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2); self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, embedding_dim)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        feat = self.global_pool(x).view(x.size(0), -1)
        return self.fc(feat)


def infonce_raw(zA, zB, tau):
    sim = torch.matmul(zA, zB.T) / tau
    tgt = torch.arange(zA.size(0), device=zA.device)
    return F.cross_entropy(sim, tgt)


def vicreg_terms(z, gamma=1.0, eps=1e-4):
    z = z - z.mean(0, keepdim=True)
    std = torch.sqrt(z.var(0) + eps)
    var_loss = torch.mean(F.relu(gamma - std))
    B, D = z.shape
    cov = (z.T @ z) / (B - 1)
    off = cov - torch.diag(torch.diag(cov))
    cov_loss = off.pow(2).sum() / D
    return var_loss, cov_loss


def vicreg_loss(zA, zB, tau, lambda_v=25.0, lambda_c=1.0):
    nce = infonce_raw(zA, zB, tau)
    vA, cA = vicreg_terms(zA); vB, cB = vicreg_terms(zB)
    return nce + lambda_v * (vA + vB) / 2 + lambda_c * (cA + cB) / 2


_e = SupEncoder(4).to(device)
print('sanity z shape:', tuple(_e(torch.randn(4, 3, 31, 51).to(device)).shape))

## Cell 5 — `run_dim(d, seed)` (locked recipe) + Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Locked recipe (Session 41): bs64 + tau0.07 + wd1e-4 + cosine, vicreg loss, full
# epochs, NO early stopping. (Run-1 bs256/wd1e-3 was a Stage-1 noise artifact;
# bs64 fills the 2-D disk more fully and gives higher phase.)
LOCKED = {'temperature': 0.07, 'batch_size': 64, 'lr': 1e-3,
          'weight_decay': 1e-4, 'epochs': 50}


def extract_emb(enc, d, idx):
    enc.eval(); out = np.zeros((len(idx), d), np.float32)
    with torch.no_grad():
        for s in range(0, len(idx), 256):
            b = torch.from_numpy(X[idx[s:s+256]]).float().to(device)
            out[s:s+256] = enc(b).cpu().numpy()
    return out


def enso_z(emb_all):
    phases = range(1, 9); obs = []
    for ph in phases:
        mEN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'El Nino')
        mLN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        obs.append(np.linalg.norm(emb_all[mEN.values].mean(0) - emb_all[mLN.values].mean(0)))
    rng = np.random.default_rng(42); base = []
    for _ in range(100):
        shuf = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values; t = []
        for ph in phases:
            mph = (labels['bsiso_phase'] == ph).values
            mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            t.append(np.linalg.norm(emb_all[mEN].mean(0) - emb_all[mLN].mean(0)))
        if t: base.append(np.mean(t))
    return float((np.mean(obs) - np.mean(base)) / (np.std(base) + 1e-8))


def run_dim(d, seed=42, epochs=None, verbose=True):
    epochs = epochs or LOCKED['epochs']
    torch.manual_seed(seed); np.random.seed(seed)
    tr = PairDataset(X, labels, phase_enso_index, 'train', train_idx)
    va = PairDataset(X, labels, phase_enso_index, 'val',   val_idx)
    use_cuda = (device.type == 'cuda')
    trl = DataLoader(tr, batch_size=LOCKED['batch_size'], shuffle=True,  num_workers=2,
                     pin_memory=use_cuda, drop_last=True)
    val = DataLoader(va, batch_size=LOCKED['batch_size'], shuffle=False, num_workers=2,
                     pin_memory=use_cuda)
    enc = SupEncoder(d).to(device)
    opt = optim.Adam(enc.parameters(), lr=LOCKED['lr'], weight_decay=LOCKED['weight_decay'])
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    for ep in range(epochs):
        enc.train()
        for fa, fb in trl:
            fa, fb = fa.to(device), fb.to(device)
            loss = vicreg_loss(enc(fa), enc(fb), LOCKED['temperature'])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0); opt.step()
        sch.step()
    emb_all = extract_emb(enc, d, np.arange(len(X)))
    m = emb_metrics(emb_all[val_active_idx])
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(emb_all[train_idx], labels.loc[train_idx, 'bsiso_phase'].values)
    phase_val = float(accuracy_score(labels.loc[val_idx, 'bsiso_phase'].values, clf.predict(emb_all[val_idx])))
    z = enso_z(emb_all)
    res = {'dim': d, 'seed': seed, 'phase_val': phase_val, 'enso_z': z,
           'eff_rank': m['eff_rank'], 'eig_ratio': m['eig_ratio'], 'norm_max': m['norm_max']}
    if verbose:
        print(f"  d={d:2d} seed={seed}: phase={phase_val*100:.1f}%  z={z:.2f}  "
              f"eff_rank={m['eff_rank']:.2f}  nmax={m['norm_max']:.1f}")
    return res, enc

print('run_dim() ready. Locked recipe:', LOCKED)

## Cell 6 — Dimension-Sweep Curve (1 seed across dims)

In [ ]:
DIMS = [1, 2, 4, 8, 16, 32, 64]
RESULTS = []
print('=== Dimension sweep (seed 42) ===')
for d in DIMS:
    r, _ = run_dim(d, seed=42)
    RESULTS.append(r)
    with open(f'{RESULTS_DIR}/dim_sweep_results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2)

df = pd.DataFrame(RESULTS)
df['phase%'] = (df['phase_val'] * 100).round(1)
print('\n', df[['dim', 'phase%', 'enso_z', 'eff_rank', 'norm_max']].to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/dim_sweep_table.csv', index=False)

## Cell 7 — Phase-vs-Dimension Plot (with 64-D baseline + NSV d̂=4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
dims = df['dim'].values

axes[0].plot(dims, df['phase%'], 'o-', lw=2, ms=8)
axes[0].axhline(67.7, color='g', ls='--', label='64-D baseline 67.7%')
axes[0].axhline(33.2, color='orange', ls='--', label='2-D L2-circle 33.2%')
axes[0].axhline(12.5, color='r', ls=':', label='random 12.5%')
axes[0].axvline(4, color='purple', ls=':', alpha=0.7, label='NSV d̂=4')
axes[0].set_xscale('log', base=2); axes[0].set_xticks(dims); axes[0].set_xticklabels(dims)
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('BSISO phase val acc (%)')
axes[0].set_title('Phase accuracy vs embedding dim', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(dims, df['enso_z'], 'o-', color='steelblue', lw=2, ms=8)
axes[1].axhline(3.83, color='g', ls='--', label='64-D z=3.83')
axes[1].axvline(4, color='purple', ls=':', alpha=0.7)
axes[1].set_xscale('log', base=2); axes[1].set_xticks(dims); axes[1].set_xticklabels(dims)
axes[1].set_xlabel('Embedding dimension'); axes[1].set_ylabel('ENSO displacement z')
axes[1].set_title('ENSO z vs embedding dim', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(dims, df['eff_rank'], 'o-', color='firebrick', lw=2, ms=8)
axes[2].plot(dims, dims, 'k:', alpha=0.4, label='eff_rank = dim (full use)')
axes[2].axvline(4, color='purple', ls=':', alpha=0.7)
axes[2].set_xscale('log', base=2); axes[2].set_yscale('log', base=2)
axes[2].set_xticks(dims); axes[2].set_xticklabels(dims)
axes[2].set_xlabel('Embedding dimension'); axes[2].set_ylabel('Effective rank')
axes[2].set_title('Effective rank vs embedding dim', fontweight='bold')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dim_sweep_curves.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved dim_sweep_curves.png')

## Cell 8 — 3-Seed Robustness at the Knee + NSV Verdict

Re-run the most informative dims (d=4 and the one just below saturation) over 3 seeds, and report where phase reaches ~90% of the 64-D ceiling — the supervised estimate of BSISO intrinsic dimension to compare against NSV d̂=4.

In [ ]:
# Estimate the 'knee': smallest dim reaching >= 90% of the 64-D phase (using the d=64 run as ceiling)
ceil_phase = float(df.loc[df['dim'] == 64, 'phase_val'].iloc[0])
knee_candidates = df[df['phase_val'] >= 0.90 * ceil_phase]['dim'].tolist()
knee = min(knee_candidates) if knee_candidates else 4
FINAL_DIMS = sorted(set([4, knee]))
print(f'64-D ceiling phase = {ceil_phase*100:.1f}%  ->  90% ceiling = {0.9*ceil_phase*100:.1f}%')
print(f'Supervised knee (smallest dim >= 90% ceiling): d = {knee}')
print(f'3-seed robustness at dims: {FINAL_DIMS}\n')

seed_rows = []
for d in FINAL_DIMS:
    for sd in [42, 1, 7]:
        r, _ = run_dim(d, seed=sd)
        seed_rows.append(r)
RESULTS_3SEED = seed_rows
with open(f'{RESULTS_DIR}/dim_sweep_3seed.json', 'w') as f:
    json.dump(RESULTS_3SEED, f, indent=2)

sdf = pd.DataFrame(seed_rows)
print('\n3-seed summary:')
for d in FINAL_DIMS:
    g = sdf[sdf['dim'] == d]
    print(f"  d={d:2d}: phase={g['phase_val'].mean()*100:.1f}%±{g['phase_val'].std()*100:.1f}  "
          f"z={g['enso_z'].mean():.2f}±{g['enso_z'].std():.2f}")

print('\n' + '=' * 60)
if knee <= 4:
    print(f'VERDICT: phase saturates by d={knee} (<=4) -> SUPPORTS NSV d̂=4. '
          f'BSISO state is ~4-D; the 2-D plateau was a dimension limit, not a training failure.')
elif knee <= 8:
    print(f'VERDICT: knee at d={knee} -> roughly consistent with NSV d̂=4 (supervised needs a bit more headroom).')
else:
    print(f'VERDICT: knee at d={knee} (>8) -> supervised contrastive needs more dims than NSV d̂=4; discuss.')
print('=' * 60)

---
## Done!

**Send back:** `dim_sweep_curves.png`, `dim_sweep_table.csv`, and the Cell-8 verdict.

Reading the result:
- If phase climbs steeply 2→4 and **flattens by d≈4** → end-to-end supervised confirmation of NSV **d̂=4**: the 2-D plateau was a dimensionality limit, not a training bug.
- If it keeps climbing past d=8 → supervised contrastive needs more capacity than the NSV manifold dimension (worth discussing — could be label noise or the probe).

Next depending on verdict: lock the d≈4 supervised encoder as the baseline for **nb08** (SSL temporal) and the three-way comparison.

---
*DDCS Project | jh9141@nyu.edu*